# STAT 8330 Old Final Exam Solutions: Fall 2022

This notebook gives complete Python solutions for the Fall 2022 old final exam. It is self-contained and intentionally verbose: each executable line is commented so the logic can be adapted quickly under timed-exam conditions.

Source PDF in this folder: `FInalExam8330_Fall2022.pdf`.

## Setup

The exam allows Python or R. These solutions use Python equivalents for the requested statistical workflows:

- simulation for order statistics,
- LOWESS/LOESS-style smoothing with cross-validation,
- randomized maximal-margin classification in two dimensions.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [scipy.stats normal distribution](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [KFold cross-validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- [statsmodels LOWESS](https://www.statsmodels.org/stable/generated/statsmodels.nonparametric.smoothers_lowess.lowess.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)

In [ ]:
import numpy as np  # Import numerical arrays and random-number generation tools.
import pandas as pd  # Import DataFrame tools for tabular output.
import matplotlib.pyplot as plt  # Import plotting tools for exam figures.
from IPython.display import display  # Import display so DataFrames render cleanly in notebooks.
from scipy import stats  # Import probability distributions for optional simulation checks.
from sklearn.model_selection import KFold  # Import k-fold cross-validation splitter.
from statsmodels.nonparametric.smoothers_lowess import lowess  # Import LOWESS smoother as Python's LOESS-style tool.
RANDOM_SEED = 123  # Store the random seed in one visible place.
rng = np.random.default_rng(RANDOM_SEED)  # Create a reproducible random-number generator.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness used by some libraries.
plt.style.use('default')  # Use a predictable plotting style.


## Problem 1: Simulating The Distribution Of An Order Statistic

**Prompt summary.** Write `ordersim(x, k, n, mu1, sd1, nrep, lower.tail)` to estimate the probability that the kth order statistic from `n` Normal observations is at most `x`. If `lower.tail=False`, return the upper-tail probability instead.

**Theory.** The kth order statistic is the kth smallest value in a sample. A simulation solution repeatedly draws samples, sorts each sample, extracts the kth value, and counts how often it falls below the threshold. The exact formula can be checked with a Binomial probability, but the requested function itself should use simulation.

In [ ]:
def ordersim(x, k, n, mu1, sd1, nrep, lower_tail=True, random_state=123):  # Define the requested order-statistic simulation function.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator for this function call.
    if k < 1 or k > n:  # Check that the requested order statistic exists.
        raise ValueError('k must be between 1 and n')  # Stop with a clear message if k is invalid.
    if sd1 <= 0:  # Check that the normal standard deviation is positive.
        raise ValueError('sd1 must be positive')  # Stop with a clear message if the standard deviation is invalid.
    samples = rng.normal(loc=mu1, scale=sd1, size=(nrep, n))  # Simulate nrep independent samples of size n from the requested Normal distribution.
    sorted_samples = np.sort(samples, axis=1)  # Sort each simulated sample from smallest to largest along each row.
    kth_values = sorted_samples[:, k - 1]  # Extract the kth order statistic using Python's zero-based indexing.
    lower_probability = np.mean(kth_values <= x)  # Estimate P(kth order statistic <= x) by the simulation fraction.
    if lower_tail:  # Check whether the lower-tail probability was requested.
        return lower_probability  # Return the estimated lower-tail probability.
    return 1.0 - lower_probability  # Return the estimated upper-tail probability for the kth order statistic.

def exact_order_stat_lower_probability(x, k, n, mu1, sd1):  # Define an optional exact check for the simulation result.
    p_single = stats.norm.cdf(x, loc=mu1, scale=sd1)  # Compute P(one Normal observation <= x).
    exact_probability = 1.0 - stats.binom.cdf(k - 1, n, p_single)  # Compute P(at least k observations are <= x).
    return exact_probability  # Return the exact lower-tail probability for comparison.


In [ ]:
simulated_lower = ordersim(  # Run the simulation for a concrete lower-tail example.
    x=0.0,  # Set the threshold for the kth order statistic.
    k=3,  # Request the third smallest observation.
    n=10,  # Use samples of size ten.
    mu1=0.0,  # Use a standard Normal mean.
    sd1=1.0,  # Use a standard Normal standard deviation.
    nrep=50000,  # Use many simulation replicates for stability.
    lower_tail=True,  # Request P(X_(k) <= x).
    random_state=RANDOM_SEED,  # Make the result reproducible.
)  # Finish the lower-tail simulation call.
simulated_upper = ordersim(  # Run the same simulation for the upper-tail probability.
    x=0.0,  # Use the same threshold.
    k=3,  # Use the same order statistic.
    n=10,  # Use the same sample size.
    mu1=0.0,  # Use the same mean.
    sd1=1.0,  # Use the same standard deviation.
    nrep=50000,  # Use the same number of simulation replicates.
    lower_tail=False,  # Request P(X_(k) > x).
    random_state=RANDOM_SEED,  # Make the result reproducible.
)  # Finish the upper-tail simulation call.
exact_lower = exact_order_stat_lower_probability(0.0, 3, 10, 0.0, 1.0)  # Compute the exact lower-tail probability as a check.
problem1_table = pd.DataFrame(  # Build a compact result table.
    [{'quantity': 'simulated lower tail', 'value': simulated_lower}, {'quantity': 'simulated upper tail', 'value': simulated_upper}, {'quantity': 'exact lower-tail check', 'value': exact_lower}]  # Store the three results as table rows.
)  # Finish creating the result table.
display(problem1_table)  # Display the simulation and exact-check results.


**Problem 1 exam takeaway.** The simulation only needs four operations: simulate samples, sort each row, select column `k - 1`, and average the indicator that the kth value is at most `x`. The exact Binomial calculation is useful only as a sanity check.

## Problem 2: Choosing The LOESS Span By K-Fold Cross-Validation

**Prompt summary.** Write `findspan(dat, smoothvec, k, plotY)` where `dat` has one predictor in column 1 and one response in column 2. The function should search candidate LOESS spans with k-fold CV, return the best span, and optionally plot the data plus the optimal smooth curve.

**Python note.** R has `loess`. In Python, the closest standard course-friendly equivalent is `statsmodels.nonparametric.smoothers_lowess.lowess`. LOWESS returns fitted values on the training x-values, so the validation predictions below use interpolation from the fitted LOWESS curve.

In [ ]:
def make_loess_demo_data(n=120, noise=0.35, random_state=123):  # Define a simulator for a one-predictor nonlinear regression example.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Simulate sorted predictor values for clean plotting.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true smooth mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add independent Gaussian noise to make observed responses.
    dat = pd.DataFrame({'x': x, 'y': y})  # Store the simulated data in the requested two-column DataFrame form.
    return dat  # Return the simulated data frame.

def lowess_predict(train_dat, x_new, span):  # Define a helper that fits LOWESS and predicts at new x values.
    train_sorted = train_dat.sort_values(train_dat.columns[0])  # Sort training data by the predictor for stable interpolation.
    x_train = train_sorted.iloc[:, 0].to_numpy()  # Extract the training predictor values.
    y_train = train_sorted.iloc[:, 1].to_numpy()  # Extract the training response values.
    fitted_curve = lowess(y_train, x_train, frac=span, return_sorted=True)  # Fit LOWESS at the requested span.
    predictions = np.interp(x_new, fitted_curve[:, 0], fitted_curve[:, 1])  # Interpolate LOWESS fitted values to validation x locations.
    return predictions  # Return the validation predictions.

def findspan(dat, smoothvec, k, plotY=True):  # Define the requested span-selection function.
    splitter = KFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)  # Create reproducible k-fold splits.
    x_all = dat.iloc[:, 0].to_numpy()  # Extract all predictor values from column one.
    y_all = dat.iloc[:, 1].to_numpy()  # Extract all response values from column two.
    rows = []  # Create a list for candidate span CV results.
    for span in smoothvec:  # Loop through the candidate smoothing spans.
        fold_errors = []  # Create a list for validation errors at this span.
        for train_index, test_index in splitter.split(dat):  # Loop through the k training-validation splits.
            train_dat = dat.iloc[train_index, :]  # Select the training rows for this fold.
            x_valid = x_all[test_index]  # Select validation predictor values for this fold.
            y_valid = y_all[test_index]  # Select validation response values for this fold.
            valid_pred = lowess_predict(train_dat, x_valid, span)  # Predict validation responses using only training data.
            fold_mse = np.mean((y_valid - valid_pred) ** 2)  # Compute validation mean squared error for this fold.
            fold_errors.append(fold_mse)  # Store the fold error.
        rows.append({'span': span, 'cv_mse': np.mean(fold_errors)})  # Store the average CV error for this span.
    cv_table = pd.DataFrame(rows)  # Convert candidate span results to a DataFrame.
    best_span = float(cv_table.loc[cv_table['cv_mse'].idxmin(), 'span'])  # Select the span with the smallest CV MSE.
    findspan.last_cv_table = cv_table  # Attach the CV table to the function for optional inspection after the call.
    if plotY:  # Check whether the caller requested the plot.
        fitted_curve = lowess(y_all, x_all, frac=best_span, return_sorted=True)  # Fit LOWESS on all data using the selected span.
        plt.figure(figsize=(6, 4))  # Create a compact plotting canvas.
        plt.scatter(x_all, y_all, s=22, alpha=0.60, label='simulated data')  # Plot the observed data points.
        plt.plot(fitted_curve[:, 0], fitted_curve[:, 1], color='red', label=f'best span = {best_span:.2f}')  # Plot the selected LOWESS curve.
        plt.xlabel(dat.columns[0])  # Label the predictor axis.
        plt.ylabel(dat.columns[1])  # Label the response axis.
        plt.title('LOESS span selected by k-fold CV')  # Title the figure.
        plt.legend()  # Show plot labels.
        plt.show()  # Render the plot.
    return best_span  # Return the optimal smoothing span as requested.


In [ ]:
loess_dat = make_loess_demo_data(n=120, noise=0.35, random_state=RANDOM_SEED)  # Simulate a nonlinear data set for the function demonstration.
candidate_spans = np.linspace(0.15, 0.85, 8)  # Define candidate spans from flexible to smooth.
best_span = findspan(loess_dat, candidate_spans, k=5, plotY=True)  # Use the requested function to choose the best span and plot the result.
display(findspan.last_cv_table)  # Display the CV table saved during the function call.
print(f'Best span selected by 5-fold CV: {best_span:.3f}')  # Print the selected span.


**Problem 2 exam takeaway.** A smoothing parameter should be chosen by validation error, not by how smooth the curve looks. Each fold must fit LOWESS without the validation fold.

## Problem 3: Randomized Maximal-Margin Classifier

**Prompt summary.** In two dimensions, generate many random separating lines, keep only those that classify all points correctly, compute each valid line's margin, and return the line with the largest margin.

**Theory note.** For a line `beta0 + beta1*x1 + beta2*x2 = 0`, the standard point-to-line distance uses denominator `sqrt(beta1^2 + beta2^2)`. The exam PDF prints a denominator including `beta0`; the code below uses the standard geometric distance because the intercept is not part of the normal vector.

In [ ]:
def make_separable_demo_data(n_per_class=35, random_state=123):  # Define a simulator for two linearly separable classes.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator.
    class0 = rng.normal(loc=[-1.5, -1.5], scale=0.35, size=(n_per_class, 2))  # Simulate class 0 around the lower-left center.
    class1 = rng.normal(loc=[1.5, 1.5], scale=0.35, size=(n_per_class, 2))  # Simulate class 1 around the upper-right center.
    x = np.vstack([class0, class1])  # Stack both classes into one predictor matrix.
    y = np.array([0] * n_per_class + [1] * n_per_class)  # Create binary class labels.
    return x, y  # Return predictors and labels.

def MaxMargClass(x, y, nlines=100, bmean=1, bsd=3, makePlot=True, random_state=123):  # Define the requested randomized max-margin function.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator.
    x = np.asarray(x)  # Convert predictors to a NumPy array.
    y_signed = np.where(np.asarray(y) == 1, 1, -1)  # Convert class labels to -1 and +1.
    projection_limit = np.max(np.abs(x)) * 4.0  # Set a broad range for random intercepts.
    best_line = None  # Initialize the best line coefficients.
    best_margin = -np.inf  # Initialize the best margin at negative infinity.
    for line_number in range(nlines):  # Generate and test many random candidate lines.
        beta1 = rng.normal(loc=bmean, scale=bsd)  # Generate a random coefficient for x1.
        beta2 = rng.normal(loc=bmean, scale=bsd)  # Generate a random coefficient for x2.
        beta0 = rng.uniform(-projection_limit, projection_limit)  # Generate a random intercept over the data range.
        beta = np.array([beta0, beta1, beta2])  # Store the line coefficients as beta0, beta1, beta2.
        scores = beta[0] + beta[1] * x[:, 0] + beta[2] * x[:, 1]  # Compute signed scores relative to the line.
        predicted_signed = np.where(scores >= 0.0, 1, -1)  # Convert scores into -1/+1 class predictions.
        if np.mean(predicted_signed == y_signed) < 0.5:  # Check whether the line orientation is mostly reversed.
            beta = -beta  # Flip the coefficients to reverse class orientation.
            scores = -scores  # Flip the scores consistently with the coefficients.
            predicted_signed = np.where(scores >= 0.0, 1, -1)  # Recompute signed predictions after flipping.
        if not np.all(predicted_signed == y_signed):  # Check whether every training point is correctly classified.
            continue  # Skip this line if it does not perfectly separate the classes.
        denominator = np.sqrt(beta[1] ** 2 + beta[2] ** 2)  # Compute the norm of the line's normal vector.
        distances = np.abs(scores) / denominator  # Compute each point's distance to the candidate line.
        margin = float(np.min(distances))  # Define the margin as the smallest point-to-line distance.
        if margin > best_margin:  # Check whether this line improves the best margin found so far.
            best_margin = margin  # Store the new best margin.
            best_line = beta  # Store the new best line coefficients.
    if best_line is None:  # Check whether the random search found no separating line.
        raise RuntimeError('No separating line was found; increase nlines or change random generation settings.')  # Give a helpful failure message.
    if makePlot:  # Check whether the caller requested a plot.
        plt.figure(figsize=(6, 4))  # Create a compact plotting canvas.
        plt.scatter(x[:, 0], x[:, 1], c=y, s=34, alpha=0.80)  # Plot the data colored by class.
        x_grid = np.linspace(x[:, 0].min() - 0.5, x[:, 0].max() + 0.5, 150)  # Create x1 values for the line plot.
        y_grid = -(best_line[0] + best_line[1] * x_grid) / best_line[2]  # Convert beta coefficients to x2 values on the line.
        plt.plot(x_grid, y_grid, color='red', label=f'margin = {best_margin:.3f}')  # Plot the best separating line.
        plt.xlabel('x1')  # Label the first predictor axis.
        plt.ylabel('x2')  # Label the second predictor axis.
        plt.title('Randomized maximal-margin classifier')  # Title the plot.
        plt.legend()  # Show the margin label.
        plt.show()  # Render the plot.
    return {'best line': best_line, 'M': best_margin}  # Return a labeled result dictionary matching the exam prompt.


In [ ]:
x3, y3 = make_separable_demo_data(n_per_class=35, random_state=RANDOM_SEED)  # Simulate linearly separable two-class data.
max_margin_result = MaxMargClass(x3, y3, nlines=8000, bmean=1, bsd=3, makePlot=True, random_state=RANDOM_SEED)  # Run the randomized max-margin search.
print('Best line coefficients beta0, beta1, beta2:')  # Label the coefficient output.
print(max_margin_result['best line'])  # Print the selected separating line coefficients.
print('Estimated margin:')  # Label the margin output.
print(max_margin_result['M'])  # Print the largest margin found.


**Problem 3 exam takeaway.** The function is intentionally inefficient, but it demonstrates the margin idea: among lines that perfectly separate the classes, prefer the one that keeps the closest point as far from the boundary as possible.